# Notebook Detaille - Gestionnaire de Budget : Finance Zen# (Version debutant - explications approfondies)---## C'est quoi ce projet ?Ce projet c'est une **application web de gestion de budget personnel**. En gros, ca te permet de :- Ajouter des depenses (courses, loyer, metro...) et des revenus (salaire, freelance...)- Voir ton solde en temps reel (combien il te reste)- Visualiser tes depenses/revenus dans un graphique en forme de donut- Filtrer par categorie (alimentation, transport, loisirs...)- Changer entre mode clair et mode sombre### Pourquoi c'est un bon exercice pour apprendre ?Parce qu'on touche a **tout** ce qu'un developpeur web junior doit savoir :| Competence | Ce qu'on apprend ici ||---|---|| **Frontend** | React, TypeScript, gestion d'etat avec useReducer || **Backend** | Creer une API REST avec Express.js || **Communication** | Faire parler le front et le back ensemble avec `fetch` || **Persistence** | Sauvegarder des donnees dans un fichier JSON || **Design** | CSS variables, theme sombre, animations, responsive || **Graphiques** | Afficher des donnees visuellement avec Chart.js |---## Avant de commencer : les prerequisSi tu debutes, voila les notions que tu dois connaitre un minimum avant de lire ce notebook :1. **HTML/CSS de base** : savoir ce qu'est une balise, un selecteur CSS, etc.2. **JavaScript de base** : variables, fonctions, tableaux, objets, boucles3. **Node.js** : avoir Node installe sur ton PC. C'est lui qui fait tourner le backend.4. **Terminal** : savoir ouvrir un terminal et taper des commandes> Si tu connais pas encore React ou TypeScript, pas de panique !> Ce notebook explique chaque concept en detail au fur et a mesure.

---# PARTIE 1 : L'architecture du projet (la structure)## C'est quoi une architecture "client-serveur" ?Imagine un restaurant :- Le **client** (toi au restaurant) = le **navigateur web** (Chrome, Firefox...)- Le **serveur** (le cuisinier en cuisine) = le **backend Express.js**- Le **menu** que tu lis = le **frontend React** (l'interface)- La **cuisine** ou on prepare = le **serveur** qui traite tes demandes- Le **frigo** ou sont les ingredients = le **fichier JSON** (la base de donnees)Quand tu cliques sur "Ajouter une transaction" dans l'interface :1. Le **frontend** (React) prepare les donnees2. Il envoie une **requete HTTP** au **backend** (Express)3. Le **backend** recoit la requete, ecrit dans le fichier JSON4. Il repond "OK, c'est fait" au frontend5. Le **frontend** met a jour l'affichage```[TON NAVIGATEUR]                    [TON PC]                                    React (interface)                   Express (serveur)   port 5173          ---HTTP-->       port 3001                                         |                                    lit/ecrit dans                                  data/transactions.json```## C'est quoi un "port" ?Un port c'est comme un numero de porte dans un immeuble. Ton PC a plein de ports.- Le frontend ecoute sur la porte **5173** (c'est Vite qui choisit ce numero)- Le backend ecoute sur la porte **3001** (c'est nous qui l'avons choisi)Quand tu vas sur `http://localhost:5173` dans ton navigateur, tu frappes a la porte 5173 = le frontend React.## L'arborescence des fichiers```gestionnaire-budget/||-- src/                    <-- FRONTEND (ce que l'utilisateur voit)|   |-- App.tsx             <-- LE fichier principal, toute la logique React|   |-- index.css           <-- Tous les styles CSS (couleurs, animations...)|   |-- main.tsx            <-- Point d'entree : "demarre React ici"||-- data/|   |-- transactions.json   <-- Notre "base de donnees" (un simple fichier)||-- server.js               <-- BACKEND (le serveur qui recoit les requetes)|-- vite.config.ts          <-- Configuration de Vite (l'outil de dev frontend)|-- package.json            <-- Liste des dependances et commandes|-- index.html              <-- La page HTML de base (presque vide)```

---# PARTIE 2 : `package.json` - La carte d'identite du projet## C'est quoi package.json ?C'est le fichier qui dit a Node.js : "voila ce dont j'ai besoin pour fonctionner". C'est comme une liste de courses : tu listes toutes les librairies (= ingredients) necessaires, et `npm install` les telecharge pour toi.```json{  "name": "gestionnaire-budget",  "type": "module",  "scripts": {    "dev": "vite",    "server": "node server.js"  },  "dependencies": {    "chart.js": "^4.4.7",    "cors": "^2.8.6",    "express": "^5.2.1",    "react": "^19.2.0",    "react-dom": "^19.2.0"  }}```## Explication de chaque ligne### `"type": "module"`Ca dit a Node.js d'utiliser la **syntaxe moderne** pour les imports :```javascript// Syntaxe MODERNE (ESM) - celle qu'on utilise avec "type": "module"import express from 'express'// Syntaxe ANCIENNE (CommonJS) - sans "type": "module"const express = require('express')```Les deux font la meme chose, mais ESM c'est le standard actuel.### `"scripts"` - Les commandes raccourcisAu lieu de taper des commandes longues, on cree des raccourcis :- `npm run dev` = execute `vite` = lance le frontend- `npm run server` = execute `node server.js` = lance le backend### `"dependencies"` - Les librairies installees| Librairie | A quoi elle sert | Taille approximative ||---|---|---|| `react` | Creer des interfaces dynamiques | Le coeur du frontend || `react-dom` | Permet a React de manipuler le HTML | Compagnon obligatoire de React || `express` | Creer un serveur HTTP facilement | Le coeur du backend || `cors` | Autoriser les requetes entre differentes origines | Petit middleware de securite || `chart.js` | Dessiner des graphiques (ici un donut) | Librairie de data visualization |### Le `^` devant les versions`"express": "^5.2.1"` signifie "installe la version 5.2.1 ou plus recente compatible".Le `^` autorise les mises a jour mineures (5.2.2, 5.3.0) mais pas les majeures (6.0.0).---## Comment lancer le projet```bash# Etape 1 : installer les dependances (a faire UNE SEULE FOIS)npm install# cette commande lit package.json et telecharge tout dans le dossier node_modules/# Etape 2 : ouvrir DEUX terminaux# Terminal 1 : lancer le backendnpm run server# -> "Serveur demarre sur http://localhost:3001"# Terminal 2 : lancer le frontendnpm run dev# -> "Local: http://localhost:5173"# Etape 3 : ouvrir http://localhost:5173 dans ton navigateur```> IMPORTANT : les deux terminaux doivent rester ouverts en meme temps !> Si tu fermes le terminal du backend, les donnees ne seront plus sauvegardees.

---# PARTIE 3 : `vite.config.ts` - Le proxy magique## C'est quoi Vite ?**Vite** (prononce "vite" comme en francais !) c'est un outil de developpement pour le frontend. Il fait plusieurs choses :- Il **compile** le TypeScript et le JSX en JavaScript que le navigateur comprend- Il **recharge automatiquement** la page quand tu modifies un fichier (hot reload)- Il **sert** les fichiers du frontend sur le port 5173## Le fichier de config```typescriptimport { defineConfig } from 'vite'import react from '@vitejs/plugin-react'export default defineConfig({    plugins: [react()],       // active le support JSX/TSX    server: {        proxy: {            '/api': 'http://localhost:3001'        }    }})```## C'est quoi le proxy et pourquoi on en a besoin ?### Le probleme : le CORSImagine que tu es dans ta maison (port 5173) et tu veux emprunter du sucre au voisin (port 3001).Pour le navigateur, deux ports differents = deux "maisons" differentes. Par securite, le navigateur **bloque** les echanges entre deux origines differentes. C'est le **CORS** (Cross-Origin Resource Sharing).### La solution : le proxyLe proxy c'est comme un tunnel secret entre tes deux maisons. Au lieu d'aller frapper chez le voisin directement, tu passes par un couloir interne.En pratique :```SANS proxy (BLOQUE par le navigateur) :  fetch('http://localhost:3001/api/transactions')  --> ERREUR CORS !AVEC proxy (ca marche) :  fetch('/api/transactions')  --> Vite intercepte les URLs qui commencent par /api  --> Il les redirige vers http://localhost:3001/api/transactions  --> Le navigateur croit que tout vient du meme endroit = pas de CORS```C'est pour ca que dans le code React, on ecrit juste `fetch('/api/transactions')` sans preciser le port. Vite s'occupe de tout en coulisses.

---# PARTIE 4 : `data/transactions.json` - La base de donnees## Pourquoi un fichier JSON ?Dans un vrai projet pro, on utiliserait une **vraie base de donnees** (PostgreSQL, MongoDB...).Mais pour apprendre, un fichier JSON c'est parfait parce que :- C'est un simple fichier texte qu'on peut ouvrir et lire- Pas besoin d'installer quoi que ce soit- Ca suffit pour un seul utilisateur## A quoi il ressemble```json[    {        "id": "1772198316422yveub4k0zl",        "name": "Courses supermarche",        "amount": 85.50,        "type": "depense",        "category": "alimentation",        "date": "2026-02-03"    },    {        "id": "1772198400000abc123",        "name": "Salaire mensuel",        "amount": 2200,        "type": "revenu",        "category": "salaire",        "date": "2026-02-01"    }]```C'est un **tableau** (les crochets `[]`) qui contient des **objets** (les accolades `{}`).Chaque objet represente une transaction.## Les champs d'une transaction expliques| Champ | Type | Exemple | Explication detaillee ||---|---|---|---|| `id` | string | `"17721983..."` | Un identifiant **unique** pour chaque transaction. Genere avec `Date.now()` + `Math.random()`. Permet de savoir QUELLE transaction supprimer. || `name` | string | `"Courses"` | La description que l'utilisateur tape dans le formulaire || `amount` | number | `85.50` | Le montant en euros. **Toujours positif**, meme pour les depenses. C'est le champ `type` qui dit si c'est une entree ou sortie. || `type` | string | `"depense"` | Soit `"revenu"` (argent qui rentre) soit `"depense"` (argent qui sort) || `category` | string | `"alimentation"` | Une des 8 categories : alimentation, transport, loisirs, logement, sante, salaire, freelance, autre || `date` | string | `"2026-02-03"` | Date au format ISO : Annee-Mois-Jour. Ce format permet de trier facilement les dates. |### Pourquoi l'id est bizarre ?L'id est genere comme ca en JavaScript :```javascriptDate.now().toString() + Math.random().toString(36).substring(2)// Date.now()    -> 1772198316422     (timestamp = nb de millisecondes depuis 1970)// Math.random() -> 0.7845...         (nombre aleatoire)// .toString(36) -> "yveub4k0zl"     (converti en base 36 = lettres + chiffres)// Resultat : "1772198316422yveub4k0zl" = quasi impossible d'avoir un doublon```

---# PARTIE 5 : `server.js` - Le Backend Express (explications ligne par ligne)C'est le fichier qui cree notre **serveur**. Un serveur c'est un programme qui tourne en permanence et qui **attend** des requetes pour y repondre.---## 5.1 - Les imports (les outils dont on a besoin)```javascriptimport express from 'express'       // le framework web (cree le serveur)import cors from 'cors'             // gestion de la securite cross-originimport fs from 'fs'                 // "file system" = lire/ecrire des fichiersimport path from 'path'             // manipuler les chemins de fichiersimport { fileURLToPath } from 'url' // convertir une URL en chemin de fichier```### C'est quoi un import ?Un import c'est comme dire "j'ai besoin de cet outil". Au lieu de tout coder nous-memes, on utilise des librairies deja faites. Par exemple :- `express` = quelqu'un a deja code tout le systeme de serveur HTTP, on le reutilise- `fs` = c'est Node.js qui le fournit gratuitement, pas besoin de l'installer## 5.2 - Configuration du serveur```javascriptconst app = express()     // on cree une "instance" d'Express = notre serveurconst PORT = 3001         // le port sur lequel il va ecouter````express()` c'est une **fonction** qui retourne un **objet serveur**. Cet objet `app` a plein de methodes : `app.get()`, `app.post()`, `app.delete()`, `app.listen()`...## 5.3 - Reconstruction de __dirname```javascriptconst __filename = fileURLToPath(import.meta.url)const __dirname = path.dirname(__filename)```### Pourquoi c'est necessaire ?En JavaScript "ancien" (CommonJS), la variable `__dirname` existait automatiquement.Elle contenait le dossier du fichier en cours, par exemple `C:\projet\`.Mais avec la syntaxe moderne (ESM, qu'on utilise avec `"type": "module"`), cette variable n'existe plus. On doit la reconstruire :```import.meta.url     -> "file:///C:/projet/server.js"    (URL du fichier)fileURLToPath(...)  -> "C:\projet\server.js"            (chemin Windows)path.dirname(...)   -> "C:\projet"                       (juste le dossier)```On a besoin de `__dirname` pour construire le chemin vers `data/transactions.json` :```javascriptconst DATA_FILE = path.join(__dirname, 'data', 'transactions.json')// -> "C:\projet\data\transactions.json"````path.join` assemble les morceaux de chemin en gerant les `/` et `\` automatiquement (important car Windows utilise `\` et Mac/Linux utilise `/`).

## 5.4 - Les middlewares```javascriptapp.use(cors())           // autorise les requetes cross-originapp.use(express.json())   // parse automatiquement le JSON des requetes```### C'est quoi un middleware ?Un middleware c'est un **filtre** que chaque requete traverse avant d'arriver a ta route.Imagine une chaine de montage dans une usine :```Requete HTTP --> [cors()] --> [express.json()] --> [ta route GET/POST/DELETE]```- `cors()` : verifie que la requete a le droit de venir de cette origine- `express.json()` : si la requete contient du JSON dans son "body", il le **parse**   (le transforme en objet JavaScript). Sans ca, `req.body` serait `undefined`.## 5.5 - Les fonctions utilitaires read/write```javascriptfunction readTransactions() {    try {        const data = fs.readFileSync(DATA_FILE, 'utf-8')  // lit le fichier        return JSON.parse(data)                            // convertit le texte en tableau JS    } catch (err) {        return []                                          // si erreur -> tableau vide    }}function writeTransactions(transactions) {    fs.writeFileSync(        DATA_FILE,        JSON.stringify(transactions, null, 4),  // convertit le tableau JS en texte JSON        'utf-8'    )}```### Decodage des fonctions**`readTransactions()` :**1. `fs.readFileSync(fichier, 'utf-8')` : lit le fichier et retourne son contenu sous forme de **texte** (string)2. `JSON.parse(texte)` : prend ce texte et le transforme en vrai **tableau JavaScript**3. Le `try/catch` : si le fichier n'existe pas ou est corrompu, au lieu de planter le serveur, on retourne un tableau vide `[]`**`writeTransactions(transactions)` :**1. `JSON.stringify(data, null, 4)` : prend le tableau JavaScript et le transforme en texte JSON   - `null` : pas de filtre sur les proprietes   - `4` : indentation de 4 espaces pour que le fichier soit lisible2. `fs.writeFileSync(fichier, contenu, 'utf-8')` : ecrit le texte dans le fichier### C'est quoi Sync vs Async ?`readFileSync` = lecture **synchrone** = le programme attend que la lecture soit finie avant de continuer.C'est simple mais ca bloque le serveur pendant la lecture.En production on utiliserait `readFile` (asynchrone) pour ne pas bloquer, mais pour un petit projet ca suffit largement.

## 5.6 - Les 4 routes de l'API REST### C'est quoi REST ?REST c'est une convention pour organiser les URLs d'une API. Chaque **verbe HTTP** correspond a une action :| Verbe HTTP | Action | URL | Ce que ca fait ||---|---|---|---|| `GET` | Lire | `/api/transactions` | Recuperer toutes les transactions || `POST` | Creer | `/api/transactions` | Ajouter une nouvelle transaction || `DELETE` | Supprimer | `/api/transactions/:id` | Supprimer UNE transaction || `DELETE` | Supprimer | `/api/transactions` | Tout effacer |### Route GET - Recuperer les transactions```javascriptapp.get('/api/transactions', (req, res) => {    const transactions = readTransactions()   // lit le fichier JSON    res.json(transactions)                    // renvoie le tableau en JSON})```C'est la plus simple : on lit, on renvoie. Le navigateur recoit un status **200 OK** (par defaut) avec le tableau JSON.- `req` = la **requete** (ce que le client envoie). Ici on ne l'utilise pas.- `res` = la **reponse** (ce qu'on renvoie au client)- `res.json(...)` = envoie une reponse au format JSON### Route POST - Ajouter une transaction```javascriptapp.post('/api/transactions', (req, res) => {    const transactions = readTransactions()    // on lit l'existant    const newTransaction = req.body            // la nouvelle transaction est dans le "body"    // si pas d'id, on en genere un    if (!newTransaction.id) {        newTransaction.id = Date.now().toString() + Math.random().toString(36).substring(2)    }    // si pas de date, on met aujourd'hui    if (!newTransaction.date) {        newTransaction.date = new Date().toISOString().split('T')[0]    }    transactions.unshift(newTransaction)       // ajoute en DEBUT de tableau    writeTransactions(transactions)            // sauvegarde    res.status(201).json(newTransaction)       // repond avec status 201 = "Created"})```**Pourquoi `unshift` au lieu de `push` ?**- `push()` ajoute a la **fin** du tableau- `unshift()` ajoute au **debut** du tableauOn veut que la plus recente soit en premier dans l'historique.**C'est quoi `req.body` ?**Quand le frontend fait un `fetch` avec `method: 'POST'`, il envoie des donnees dans le "corps" de la requete. Grace au middleware `express.json()`, ces donnees sont automatiquement parsees et disponibles dans `req.body`.### Route DELETE /:id - Supprimer une transaction```javascriptapp.delete('/api/transactions/:id', (req, res) => {    const transactions = readTransactions()    const filtered = transactions.filter(t => t.id !== req.params.id)    if (filtered.length === transactions.length) {        return res.status(404).json({ error: 'Transaction pas trouvee' })    }    writeTransactions(filtered)    res.json({ message: 'Transaction supprimee' })})```**C'est quoi `:id` dans l'URL ?**C'est un **parametre dynamique**. Si le frontend appelle `DELETE /api/transactions/abc123`, alors `req.params.id` vaudra `"abc123"`.**Comment la suppression fonctionne :**1. On lit toutes les transactions2. On **filtre** : on garde toutes celles dont l'id est DIFFERENT de celui a supprimer3. Si le tableau filtre a la meme longueur = on n'a rien supprime = l'id n'existait pas = erreur 4044. Sinon on sauvegarde le tableau filtre (sans la transaction supprimee)### Route DELETE - Tout effacer```javascriptapp.delete('/api/transactions', (req, res) => {    writeTransactions([])    // ecrit un tableau vide dans le fichier    res.json({ message: 'Toutes les transactions ont ete supprimees' })})```## 5.7 - Demarrage du serveur```javascriptapp.listen(PORT, () => {    console.log(`Serveur demarre sur http://localhost:${PORT}`)})````app.listen(3001)` dit a Express : "ecoute sur le port 3001 et attends des requetes".La fonction callback s'execute une fois que le serveur est pret.

---# PARTIE 6 : `src/main.tsx` - Le demarrage de React```typescriptimport { StrictMode } from 'react'import { createRoot } from 'react-dom/client'import './index.css'import App from './App.tsx'createRoot(document.getElementById('root')!).render(    <StrictMode>        <App />    </StrictMode>,)```## Explication pour debutant### C'est quoi `.tsx` ?C'est du **TypeScript** + **JSX**. - **TypeScript** = JavaScript avec des **types** (on dit "cette variable est un nombre", "celle-la est un texte"...)- **JSX** = une syntaxe qui permet d'ecrire du HTML directement dans le JavaScript### Ligne par ligne1. **`import { StrictMode } from 'react'`** : on importe le mode strict de React2. **`import { createRoot } from 'react-dom/client'`** : on importe la fonction qui "branche" React sur le HTML3. **`import './index.css'`** : on charge les styles CSS (c'est Vite qui gere ca)4. **`import App from './App.tsx'`** : on importe notre composant principal5. **`document.getElementById('root')`** : dans `index.html`, il y a une `<div id="root"></div>`.    C'est la que React va injecter toute l'application.6. **Le `!`** : c'est une assertion TypeScript. Ca dit "je SAIS que cet element existe".   Sans le `!`, TypeScript dirait "et si `getElementById` retourne `null` ?"7. **`<StrictMode>`** : un outil de developpement. Il execute certains effets **deux fois**    expres pour detecter des bugs. Ca ne change rien en production.8. **`<App />`** : c'est notre composant principal. Tout part de la.

---# PARTIE 7 : `src/App.tsx` - Le coeur de l'applicationC'est LE fichier le plus important. Il fait ~500 lignes et contient toute la logique de l'interface. On va le decouper morceau par morceau.---## 7.1 - Les imports et l'enregistrement de Chart.js```typescriptimport { useReducer, useEffect, useRef, useState, useCallback } from 'react'import { Chart, ArcElement, Tooltip, Legend, DoughnutController } from 'chart.js'Chart.register(ArcElement, Tooltip, Legend, DoughnutController)```### C'est quoi les hooks React ?Les hooks sont des **fonctions speciales** fournies par React qui permettent d'ajouter des fonctionnalites a nos composants. Leur nom commence toujours par `use`.| Hook | En une phrase | Analogie ||---|---|---|| `useState` | Memorise une valeur qui peut changer | Un post-it qu'on peut effacer et reecrire || `useReducer` | Comme useState mais pour des etats complexes | Un carnet avec des regles de gestion || `useEffect` | Execute du code quand quelque chose change | Un robot qui reagit a des evenements || `useRef` | Garde une reference qui ne declenche PAS de re-render | Un tiroir secret || `useCallback` | Memorise une fonction pour ne pas la recreer | Un raccourci clavier |### Pourquoi `Chart.register(...)` ?Chart.js est **modulaire** : il ne charge pas tout par defaut pour etre plus leger.On doit lui dire manuellement quels composants on utilise :- `ArcElement` : les arcs du donut- `Tooltip` : les info-bulles au survol- `Legend` : la legende en bas- `DoughnutController` : le controleur qui dessine le donut## 7.2 - Les types TypeScript```typescriptconst CATEGORIES = [    'alimentation', 'transport', 'loisirs', 'logement',    'sante', 'salaire', 'freelance', 'autre',] as consttype Category = typeof CATEGORIES[number]type TransactionType = 'revenu' | 'depense'interface Transaction {    id: string    name: string    amount: number    type: TransactionType    category: Category    date: string}interface BudgetState {    transactions: Transaction[]}```### C'est quoi les types et pourquoi c'est utile ?En JavaScript normal, tu peux mettre n'importe quoi dans une variable. Ca cause des bugs :```javascript// JavaScript normal - pas d'erreur mais ca plante a l'executionlet montant = "cinquante"let total = montant + 10    // "cinquante10" ... oups```Avec TypeScript, le compilateur te previent AVANT :```typescript// TypeScript - erreur directe dans l'editeurlet montant: number = "cinquante"  // ERREUR : string n'est pas un number !```### `as const` - c'est quoi ?```typescriptconst CATEGORIES = ['alimentation', 'transport'] as const```Sans `as const`, TypeScript voit ca comme un `string[]` (tableau de strings quelconques).Avec `as const`, TypeScript sait que les **seules valeurs possibles** sont 'alimentation' et 'transport'.Ca empeche d'utiliser une categorie qui n'existe pas.### `interface` - c'est quoi ?Une interface c'est un **contrat**. Ca dit "un objet Transaction DOIT avoir ces champs avec ces types". Si tu oublies un champ ou tu te trompes de type, TypeScript t'alerte.

## 7.3 - Le Reducer (le gestionnaire d'etat)### C'est quoi un Reducer ?Imagine un **distributeur automatique** :- Tu inseres une piece et tu appuies sur un bouton (= tu **dispatches une action**)- La machine traite ta demande selon le bouton appuye (= le **reducer** selon le type d'action)- La machine te donne le produit et change son inventaire (= le **nouvel etat**)Le reducer c'est une **fonction pure** : meme entree = meme sortie, toujours.### Le code```typescripttype BudgetAction =    | { type: 'ADD_TRANSACTION';   payload: Transaction }    | { type: 'DELETE_TRANSACTION'; payload: string }    | { type: 'LOAD_TRANSACTIONS'; payload: Transaction[] }    | { type: 'RESET_TRANSACTIONS' }function budgetReducer(state: BudgetState, action: BudgetAction): BudgetState {    switch (action.type) {        case 'ADD_TRANSACTION': {            const updated = [action.payload, ...state.transactions]            return { transactions: updated }        }        case 'DELETE_TRANSACTION': {            const updated = state.transactions.filter(t => t.id !== action.payload)            return { transactions: updated }        }        case 'LOAD_TRANSACTIONS':            return { transactions: action.payload }        case 'RESET_TRANSACTIONS': {            return { transactions: [] }        }        default:            return state    }}```### Decodage de chaque action**`ADD_TRANSACTION`** :```typescriptconst updated = [action.payload, ...state.transactions]```Le `...` c'est le **spread operator** (operateur de decomposition). Ca "etale" le tableau.```javascript// Avant :  state.transactions = [A, B, C]// Action : action.payload = NOUVEAU// Apres :  [NOUVEAU, ...state.transactions] = [NOUVEAU, A, B, C]```On met le nouveau en premier pour que les plus recentes soient en haut de la liste.**`DELETE_TRANSACTION`** :```typescriptconst updated = state.transactions.filter(t => t.id !== action.payload)````filter` cree un **nouveau tableau** en gardant seulement les elements qui passent le test.```javascript// transactions = [{id: "abc"}, {id: "def"}, {id: "ghi"}]// action.payload = "def"// filter garde tout sauf "def"// resultat = [{id: "abc"}, {id: "ghi"}]```**`LOAD_TRANSACTIONS`** : remplace tout par les donnees du serveur**`RESET_TRANSACTIONS`** : retourne un tableau vide### Pourquoi `useReducer` et pas `useState` ?| Critere | `useState` | `useReducer` ||---|---|---|| Nombre d'operations | 1-2 | 3+ (ajouter, supprimer, charger, reset) || Complexite | Simple | Complexe mais organise || Test | Logique dans le composant | Logique isolee = facile a tester || Lisibilite | Bien pour un boolean | Bien pour des listes avec operations |

## 7.4 - Les hooks dans le composant App```typescriptfunction App() {    // --- GESTION DU THEME ---    const [theme, setTheme] = useState<'light' | 'dark'>(() => {        const saved = localStorage.getItem(THEME_KEY)        return (saved === 'dark') ? 'dark' : 'light'    })    // --- ETAT PRINCIPAL ---    const [state, dispatch] = useReducer(budgetReducer, { transactions: [] })    const [filter, setFilter] = useState<Category | 'tout'>('tout')    const [chartView, setChartView] = useState<'depenses' | 'revenus'>('depenses')    // --- CHAMPS DU FORMULAIRE ---    const [name, setName] = useState('')    const [amount, setAmount] = useState('')    const [type, setType] = useState<TransactionType>('depense')    const [category, setCategory] = useState<Category>('alimentation')    // --- REFERENCES ---    const chartRef = useRef<HTMLCanvasElement>(null)    const chartInstance = useRef<Chart | null>(null)```### `useState` en detail```typescriptconst [theme, setTheme] = useState('light')//     ^        ^                    ^//     |        |                    |//  valeur   fonction          valeur initiale// actuelle  pour la//           modifier```Chaque `useState` retourne un **tableau de 2 elements** :1. La **valeur actuelle** (ici `theme` vaut `'light'` ou `'dark'`)2. La **fonction pour modifier** cette valeur (ici `setTheme`)> IMPORTANT : quand on appelle `setTheme('dark')`, React **re-render** le composant > (il re-execute toute la fonction `App()` pour mettre a jour l'affichage).### Le "lazy initializer" du theme```typescriptconst [theme, setTheme] = useState(() => {    const saved = localStorage.getItem(THEME_KEY)    return (saved === 'dark') ? 'dark' : 'light'})```Au lieu de passer une valeur directe, on passe une **fonction**. Cette fonction :1. Regarde dans le `localStorage` si un theme a ete sauvegarde2. Si oui et c'est `'dark'`, on demarre en sombre3. Sinon, on demarre en clair**Pourquoi une fonction et pas directement la valeur ?**Parce que `localStorage.getItem()` fait un acces disque. Si on le met directement, il serait execute a CHAQUE render. Avec la fonction, il n'est execute qu'au PREMIER render.### `useRef` - les references```typescriptconst chartRef = useRef<HTMLCanvasElement>(null)const chartInstance = useRef<Chart | null>(null)````useRef` c'est comme un **tiroir** : on peut mettre quelque chose dedans et le recuperer plus tard, SANS que React re-render le composant.- `chartRef` : pointe vers l'element `<canvas>` du DOM (ou Chart.js dessine)- `chartInstance` : garde l'objet Chart.js en memoire pour pouvoir le detruireLa difference avec `useState` : modifier un `ref` ne declenche PAS de re-render.

## 7.5 - Les useEffect (les effets de bord)### C'est quoi un "effet de bord" ?Un effet de bord c'est tout ce qui se passe **en dehors** du simple affichage :- Appeler une API- Modifier le DOM directement- Sauvegarder dans localStorage- Creer un graphiqueReact gere l'affichage, mais il ne sait pas faire ces choses la tout seul. `useEffect` sert a ca.### Syntaxe de base```typescriptuseEffect(() => {    // code a executer}, [dependances])```- La **fonction** : le code a executer- Les **dependances** (le tableau a la fin) : QUAND re-executer ce code| Dependances | Quand ca s'execute ||---|---|| `[]` (vide) | Une seule fois, au premier affichage || `[theme]` | A chaque fois que `theme` change || `[a, b]` | A chaque fois que `a` OU `b` change || pas de tableau | A CHAQUE render (a eviter !) |### Effect 1 : appliquer le theme```typescriptuseEffect(() => {    document.documentElement.setAttribute('data-theme', theme)    localStorage.setItem(THEME_KEY, theme)}, [theme])```Quand `theme` change :1. On ajoute `data-theme="dark"` sur la balise `<html>`2. En CSS, le selecteur `[data-theme="dark"]` change toutes les variables de couleur3. On sauvegarde dans `localStorage` pour retrouver le choix au prochain chargement### Effect 2 : charger les transactions au demarrage```typescriptuseEffect(() => {    loadTransactions().then(loaded => {        dispatch({ type: 'LOAD_TRANSACTIONS', payload: loaded })    })}, [])   // tableau vide = une seule fois```Le flux au demarrage :1. React monte le composant (premier affichage)2. Le `useEffect` se declenche3. `loadTransactions()` fait un `fetch GET /api/transactions`4. Vite redirige vers Express (grace au proxy)5. Express lit `transactions.json` et renvoie les donnees6. On dispatche `LOAD_TRANSACTIONS` = le state se remplit7. React re-render = les transactions s'affichent### Effect 3 : le graphique Chart.jsC'est le plus complexe : a chaque changement des transactions ou du theme, on detruit l'ancien graphique et on en cree un nouveau. On le verra dans la section Chart.js.

## 7.6 - Les calculs derives (solde, totaux, filtres)Ces calculs sont recalcules a **chaque render**. C'est normal : ils sont rapides et dependent directement du state.### Calcul des totaux```typescriptconst totalRevenu = state.transactions    .filter(t => t.type === 'revenu')    .reduce((sum, t) => sum + t.amount, 0)const totalDepense = state.transactions    .filter(t => t.type === 'depense')    .reduce((sum, t) => sum + t.amount, 0)const solde = totalRevenu - totalDepense```### Comment `filter` + `reduce` marchent ensemblePrenons un exemple concret :```Transactions : [  { name: "Salaire",  amount: 2200, type: "revenu" },  { name: "Loyer",    amount: 650,  type: "depense" },  { name: "Courses",  amount: 85,   type: "depense" },  { name: "Freelance", amount: 350, type: "revenu" },]Etape 1 - filter les revenus :  [{ Salaire, 2200 }, { Freelance, 350 }]Etape 2 - reduce (accumuler) :  sum = 0  sum = 0 + 2200 = 2200    (premiere iteration)  sum = 2200 + 350 = 2550  (deuxieme iteration)  => totalRevenu = 2550Meme chose pour les depenses :  => totalDepense = 735Solde = 2550 - 735 = 1815 EUR```### Filtrage de l'historique```typescriptconst filteredTransactions = filter === 'tout'    ? state.transactions    : state.transactions.filter(t => t.category === filter)```C'est un **operateur ternaire** : `condition ? si_vrai : si_faux`- Si le filtre est sur "tout" : on montre toutes les transactions- Sinon : on garde uniquement celles de la categorie selectionnee

## 7.7 - Les handlers (fonctions de gestion des evenements)### handleSubmit - Ajouter une transaction```typescriptasync function handleSubmit(e: React.FormEvent) {    e.preventDefault()     // empeche le rechargement de la page    if (!name.trim() || !amount || parseFloat(amount) <= 0) return  // validation    const newTransaction: Transaction = {        id: Date.now().toString() + Math.random().toString(36).substring(2),        name: name.trim(),        amount: parseFloat(parseFloat(amount).toFixed(2)),        type,        category,        date: new Date().toISOString().split('T')[0],    }    try {        await fetch(API_URL, {            method: 'POST',            headers: { 'Content-Type': 'application/json' },            body: JSON.stringify(newTransaction),        })        dispatch({ type: 'ADD_TRANSACTION', payload: newTransaction })        setName('')        setAmount('')    } catch {        console.error('Erreur lors de l\'ajout')    }}```### Decodage du handleSubmit1. **`e.preventDefault()`** : par defaut, quand un formulaire est soumis, le navigateur    recharge la page. On veut pas ca (on perdrait tout l'etat React). Cette ligne l'empeche.2. **Validation** : on verifie que le nom n'est pas vide et le montant est positif.   `name.trim()` enleve les espaces autour ("  test  " -> "test").3. **Construction de l'objet** : on cree un nouvel objet Transaction avec un id unique,    le nom nettoye, le montant arrondi a 2 decimales, etc.4. **Envoi au serveur** :   ```typescript   await fetch(API_URL, {       method: 'POST',                                // verbe HTTP       headers: { 'Content-Type': 'application/json' }, // dit au serveur que c'est du JSON       body: JSON.stringify(newTransaction),           // les donnees en texte JSON   })   ```5. **Mise a jour locale** : `dispatch` ajoute la transaction dans le state React6. **Reset du formulaire** : on vide les champs### C'est quoi `async/await` ?`async/await` c'est une facon d'ecrire du code **asynchrone** (qui prend du temps) de maniere lisible :```typescript// SANS async/await (version "then") :fetch(url).then(response => {    response.json().then(data => {        console.log(data)    })})// AVEC async/await (meme resultat, plus lisible) :const response = await fetch(url)const data = await response.json()console.log(data)````await` dit "attends que cette operation soit finie avant de continuer".On doit mettre `async` devant la fonction pour pouvoir utiliser `await` dedans.### Le pattern "Optimistic Update"On met a jour l'interface AVANT d'avoir la confirmation du serveur. C'est plus rapide pour l'utilisateur (pas d'attente). Si le serveur echoue, le pire c'est que la transaction s'affiche mais ne sera pas la au prochain chargement.

---# PARTIE 8 : `src/index.css` - Le Design## 8.1 - Les variables CSS (Custom Properties)```css:root {    --bg-primary: #f5efe6;           /* fond beige */    --text-primary: #3e2c1c;         /* texte brun fonce */    --accent: #8b6f47;               /* couleur principale */    --income-color: #2e8b57;         /* vert pour les revenus */    --expense-color: #c0392b;        /* rouge pour les depenses */    --panel-radius: 14px;            /* arrondi des cartes */}```### C'est quoi `:root` ?`:root` c'est un selecteur CSS qui cible l'element **racine** du document (= `<html>`).Les variables declarees ici sont accessibles **partout** dans le CSS.### Comment utiliser une variable ?```css.panel {    background: var(--bg-card);       /* utilise la variable */    border-radius: var(--panel-radius);}````var(--nom)` recupere la valeur de la variable. L'avantage : si on change la variable, TOUS les elements qui l'utilisent changent automatiquement.## 8.2 - Le systeme de theme clair/sombre```css:root { --bg-primary: #f5efe6; }                /* clair par defaut */[data-theme="dark"] { --bg-primary: #1a1410; }  /* sombre quand data-theme="dark" */```Quand React ajoute `data-theme="dark"` sur `<html>`, les variables CSS sont **ecrasees** par les nouvelles valeurs. Tout le design change instantanement !## 8.3 - Le glassmorphisme```css.panel {    background: rgba(210, 190, 165, 0.35);   /* semi-transparent */    backdrop-filter: blur(12px);              /* flou derriere */    border: 1px solid rgba(160, 130, 95, 0.25);    border-radius: 14px;}```L'effet "verre depoli" est obtenu en combinant :- Un fond **semi-transparent** (le `0.35` dans rgba = 35% d'opacite)- Un **flou** de l'arriere-plan (`backdrop-filter: blur`)- Une **bordure fine** pour delimiter le panneau## 8.4 - Les animations```css@keyframes slideIn {    from { opacity: 0; transform: translateY(-8px); }    to   { opacity: 1; transform: translateY(0); }}.transaction-item {    animation: slideIn 0.3s ease forwards;}````@keyframes` definit une animation avec un etat de **depart** (`from`) et un etat d'**arrivee** (`to`). Ici la transaction apparait en glissant de haut en bas avec un fondu.- `0.3s` : duree de 300 millisecondes- `ease` : acceleration douce (commence lentement, accelere, ralentit)- `forwards` : l'element reste dans l'etat final apres l'animation## 8.5 - Le responsive design```css.main-grid {    display: grid;    grid-template-columns: 1fr 1fr;    /* 2 colonnes egales */}@media (max-width: 700px) {    .main-grid {        grid-template-columns: 1fr;    /* 1 seule colonne */    }}````@media` est une **media query**. Elle applique des styles uniquement si la condition est remplie. Ici : si l'ecran fait moins de 700px de large (= mobile), on passe a une seule colonne.

---# PARTIE 9 : Le JSX - La partie visuelle (le `return` d'App)## C'est quoi le JSX ?Le JSX c'est du HTML ecrit directement dans le JavaScript. React transforme ce JSX en vrais elements HTML que le navigateur affiche.```tsx// Ca c'est du JSX :<div className="panel">    <h2>Nouvelle transaction</h2></div>// React le transforme en :React.createElement('div', { className: 'panel' },    React.createElement('h2', null, 'Nouvelle transaction'))```> Note : en JSX on ecrit `className` au lieu de `class` (car `class` est un mot > reserve en JavaScript).## La structure du JSX de notre app```<div id="app">    |    |-- <header>           : logo + bouton theme    |    |-- <div balance-card> : carte avec le solde    |    |-- <div summary-row>  : revenus et depenses cote a cote    |    |-- <div main-grid>    : grille 2 colonnes    |   |-- <div panel>    : formulaire d'ajout    |   |-- <div panel>    : graphique donut    |    |-- <div panel>        : historique des transactions        |-- filtres par categorie        |-- liste des transactions```## Le rendu conditionnel```tsx{state.transactions.length > 0 && (    <button onClick={handleReset}>Tout effacer</button>)}```Le `&&` c'est un raccourci : "si la condition a gauche est vraie, affiche ce qui est a droite".Ici : le bouton "Tout effacer" n'apparait QUE s'il y a des transactions.```tsx{filteredTransactions.length === 0 ? (    <p>Aucune transaction</p>) : (    <ul>...</ul>)}```L'operateur ternaire pour afficher soit le message "aucune transaction", soit la liste.## Le `.map()` pour les listes```tsx{filteredTransactions.map(t => (    <li key={t.id} className="transaction-item">        <span>{t.name}</span>        <span>{formatMoney(t.amount)}</span>    </li>))}````.map()` transforme chaque element du tableau en JSX. C'est comme une boucle `for` mais qui retourne du HTML pour chaque element.Le `key={t.id}` est **obligatoire** : React l'utilise pour savoir quel element a change quand la liste se met a jour (performance).

---# PARTIE 10 : Resume des concepts## Le flux complet d'ajout d'une transaction```1. L'utilisateur remplit le formulaire et clique "Ajouter"2. handleSubmit() est appele3. On valide les champs (pas vides, montant > 0)4. On cree un objet Transaction avec un id unique5. fetch() envoie un POST /api/transactions avec les donnees6. Vite intercepte /api et redirige vers Express port 30017. Express recoit la requete, lit transactions.json8. Express ajoute la nouvelle transaction au debut du tableau9. Express ecrit le nouveau tableau dans transactions.json10. Express repond avec status 201 (Created)11. dispatch() met a jour le state React local12. React re-render le composant13. La nouvelle transaction apparait dans la liste avec une animation14. Le graphique et les totaux sont recalcules automatiquement```## Tableau de reference rapide### Tous les fichiers et leur role| Fichier | Role | Technologie ||---|---|---|| `package.json` | Liste des dependances + scripts | Node.js / npm || `vite.config.ts` | Config Vite + proxy API | Vite || `index.html` | Page HTML de base (quasi vide) | HTML || `server.js` | Serveur API REST (backend) | Express.js || `data/transactions.json` | Base de donnees (fichier) | JSON || `src/main.tsx` | Point d'entree React | React || `src/App.tsx` | Toute la logique frontend | React + TypeScript || `src/index.css` | Tous les styles | CSS |### Les commandes utiles```bashnpm install          # installer les dependancesnpm run dev          # lancer le frontend (Vite, port 5173)npm run server       # lancer le backend (Express, port 3001)# tester l'API avec curl :curl http://localhost:3001/api/transactions                    # GET toutcurl -X POST http://localhost:3001/api/transactions \     -H "Content-Type: application/json" \     -d '{"name":"Test","amount":42,"type":"depense","category":"loisirs"}'curl -X DELETE http://localhost:3001/api/transactions/MON_ID   # supprimer unecurl -X DELETE http://localhost:3001/api/transactions           # tout effacer```